In [2]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import pickle
import time
import numpy as np
import multiprocessing
from concurrent.futures import ProcessPoolExecutor

# Import the Worker function from your external file!
# This ensures Windows child processes can locate it successfully
from scan_utils import * 

# =================================================================
# Main execution logic
# =================================================================
if __name__ == '__main__':
    sample_num = 1000
    inputs = [(np.random.uniform(0.01, 1.5), np.random.uniform(0.01, 1.5)) for _ in range(sample_num)]
    output_dict = []
    
    # Intelligently identify the number of cores (compatible with local Windows and Koa SLURM)
    slurm_cores = os.environ.get("SLURM_CPUS_PER_TASK")
    if slurm_cores is not None:
        n_cores = int(slurm_cores)
    else:
        # For local execution, reserve one core to prevent system freeze
        n_cores = max(1, multiprocessing.cpu_count() - 1)
        
    print(f"🚀 Starting parallel computation, processing {sample_num} sample points using {n_cores} core(s)...")
    t0 = time.time()
    
    with ProcessPoolExecutor(max_workers=n_cores) as executor:
        results = executor.map(worker_task, inputs)
        
        for i, res in enumerate(results):
            if res["success"]:
                output_dict.append({
                    "current": res["current"],
                    "parameters": res["parameters"]
                })
                print(f"  ✓ Sample {i+1}/{sample_num} computation completed")
            else:
                print(f"  ✗ Sample {i+1}/{sample_num} failed: {res['error']}")
                
    t1 = time.time()
    print("="*50)
    print(f"✅ Parallel scan completed! Total wall time: {t1-t0:.4f} seconds")
    print(f"Number of valid data points: {len(output_dict)}")
    print("="*50)
    
    with open("../../results/scan_parameters.pkl", "wb") as f:
        pickle.dump({
            "output_dict": output_dict,
        }, f)
        print("Results saved to ../../results/scan_parameters.pkl")

🚀 Starting parallel computation, processing 1000 sample points using 11 core(s)...
  ✓ Sample 1/1000 computation completed
  ✓ Sample 2/1000 computation completed
  ✓ Sample 3/1000 computation completed
  ✓ Sample 4/1000 computation completed
  ✓ Sample 5/1000 computation completed
  ✓ Sample 6/1000 computation completed
  ✓ Sample 7/1000 computation completed
  ✓ Sample 8/1000 computation completed
  ✓ Sample 9/1000 computation completed
  ✓ Sample 10/1000 computation completed
  ✓ Sample 11/1000 computation completed
  ✓ Sample 12/1000 computation completed
  ✓ Sample 13/1000 computation completed
  ✓ Sample 14/1000 computation completed
  ✓ Sample 15/1000 computation completed
  ✓ Sample 16/1000 computation completed
  ✓ Sample 17/1000 computation completed
  ✓ Sample 18/1000 computation completed
  ✓ Sample 19/1000 computation completed
  ✓ Sample 20/1000 computation completed
  ✓ Sample 21/1000 computation completed
  ✓ Sample 22/1000 computation completed
  ✓ Sample 23/1000 compu